# YSSY Wind Forecast — Complete Workflow

**Source folder:** `team-members/kecheng-zhang/scripts/YSSY-code/`

```
Raw BOM fixed-width files  (RawDataC/)
        │
        ▼  data-processing/process_data.py  [STAGE 0]
  ProcessedData/*.txt
        │
        ▼  data-processing/merge_stations.py
  Station merge
        │
        ▼  data-processing/plot_availability.py.py  [DIAGNOSTIC A]
  Availability plot (raw)
        │
        ▼  data-processing/interpolateData0.5.py
  Single-gap fill (30 min)
        │
        ▼  data-processing/outage_lengths.py  [DIAGNOSTIC B]
  Outage distribution plots
        │
        ▼  data-processing/convertWindComponents.py
  Wind speed / dir -> U / V
        │
        ▼  data-processing/interpolate.py
  Spline fill (gaps <= 2.5 hr)
        │
        ▼  data-processing/plotTotalAvailability.py  [DIAGNOSTIC D]
  Per-element + concurrent 24H completeness (post-spline)
        │
        ▼  data-processing/drop_columns.py
  Drop unwanted columns
        │
        ▼  data-processing/crop_to_years.py
  Crop to [START_YEAR, END_YEAR]
        │
        ▼  data-processing/find_missing_timestamp.py  [DIAGNOSTIC C]
  Missing-timestamp check
        │
        ▼  PARQUET SAVE
  parquet/yssy/
        │
        ▼  data-processing/preprocess_data.py
  Feature engineering + train/val/test split  ->  parquet/yssy/ml/
        │
        ├─> model/YSSY_winds_24hr.py        [Option A - quick]
        ├─> model/YSSY_LightGMB.py          [Option B - tuned]
        └─> model/train_final_individual.py [Option C - per-target]
                   ^ tuned by model/tune_single_model.py
        │
        ▼  model/evaluate_all_24hr_models.py
  Evaluation (MAE / MSE per horizon)
        │
        ▼  model/plot_samples_24hr.py  +  model/plot_test_samples.py
  Forecast visualisation
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.interpolate import UnivariateSpline
import joblib, random, glob, time
from pathlib import Path
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')
print(f'pandas {pd.__version__}  |  lightgbm {lgb.__version__}')
print('All imports OK.')

In [ ]:
YSSY_CODE_DIR = Path('../../kecheng-zhang/scripts/YSSY-code')
RAW_TXT_DIR   = YSSY_CODE_DIR / 'ProcessedData'
NOTEBOOK_DIR  = Path('.')
PARQUET_DIR   = NOTEBOOK_DIR / 'parquet' / 'yssy'
MODEL_DIR     = NOTEBOOK_DIR / 'models'
PLOTS_DIR     = NOTEBOOK_DIR / 'plots' / 'yssy'
for d in [PARQUET_DIR, MODEL_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WEATHER_PARAMS       = ['air_temp','dew_point','wind_speed','wind_dir','max_gust_speed','msl_pressure','aws_flag']
INTERPOLATE_ELEMENTS = ['air_temp','dew_point','wind_speed','wind_dir','max_gust_speed','msl_pressure']
SPLINE_ELEMENTS      = ['air_temp','dew_point','msl_pressure','u_component','v_component']
COLUMNS_TO_DROP      = ['max_gust_speed','aws_flag','wind_dir_recalc']
MERGE_PAIRS          = [('YSRI Old.txt','YSRI.txt','YSRI')]
START_YEAR    = 2000
END_YEAR      = 2024
MAX_GAP_STEPS = 5
SPLINE_ORDER  = 3
SPLINE_SMOOTH = 1

# ML params — from data-processing/preprocess_data.py
TARGET_STATION       = 'YSSY'
STATIONS             = ['BELL','MTB','YBTH','YCNK','YSBK','YSCN','YSNW','YSRI','YSSY','YSWG']
STATIONS_NO_PRESSURE = ['BELL','MTB']
ALL_FILE_FEATURES    = ['air_temp','dew_point','msl_pressure','u_component','v_component']
LOOKBACK_STEPS       = 48
FORECAST_STEPS       = 48
TRAIN_PROP, VAL_PROP, TEST_PROP = 0.8, 0.1, 0.1
TIMESTAMP_COL        = 'timestamp_t'

LGBM_PARAMS = {
    'objective':'regression_l2','metric':'l2',
    'n_estimators':800,'learning_rate':0.02130325383067257,
    'num_leaves':200,'max_depth':17,'min_child_samples':45,
    'subsample':0.8767980400385894,'colsample_bytree':0.6947769061520032,
    'reg_alpha':1.959826832632918,'reg_lambda':0.12507898430314643,
    'random_state':42,'n_jobs':-1,'verbose':-1,
}
print('Configuration loaded.')
txts = list(RAW_TXT_DIR.glob('*.txt')) if RAW_TXT_DIR.exists() else []
print(f'Found {len(txts)} station file(s)' if txts else f'WARNING: {RAW_TXT_DIR} not found')

---
## Stage 0 — Raw BOM Data Ingestion
**Source:** `data-processing/process_data.py`

Reads raw BOM fixed-width `.txt` files from `RawDataC/` and writes clean CSV files
to `ProcessedData/` (one per station).

> Requires raw files in `RawDataC/`. Skip if `ProcessedData/*.txt` already exist.

In [ ]:
# SOURCE: data-processing/process_data.py

_COL_SPECS = [
    (0,2),(3,9),(10,14),(15,17),(18,20),(21,23),(24,26),
    (27,32),(33,34),(35,40),(41,42),(43,48),(49,50),
    (51,54),(55,56),(57,62),(63,64),(65,66),(67,68),
    (69,70),(71,72),(73,74),(75,76),(77,78),(79,80),
    (81,87),(88,89),(90,92),(93,94),
]
_COL_NAMES = [
    'record_id','station_id','year','month','day','hour','minute',
    'air_temp','q_air_temp','dew_point','q_dew_point',
    'wind_speed','q_wind_speed','wind_dir','q_wind_dir',
    'max_gust_speed','q_max_gust_speed',
    'cloud1_amt','q_cloud1_amt','cloud2_amt','q_cloud2_amt',
    'cloud3_amt','q_cloud3_amt','cloud4_amt','q_cloud4_amt',
    'msl_pressure','q_msl_pressure','aws_flag','end_indicator',
]
_NUM_COLS  = ['air_temp','dew_point','wind_speed','wind_dir','max_gust_speed','msl_pressure','aws_flag']
_FINAL_COLS = ['air_temp','dew_point','wind_speed','wind_dir','msl_pressure']

def ingest_raw_bom_file(filepath, out_dir):
    filepath, out_dir = Path(filepath), Path(out_dir)
    df = pd.read_fwf(filepath, colspecs=_COL_SPECS, names=_COL_NAMES, dtype=str, skiprows=1)
    if df.empty: print(f'  {filepath.name}: empty'); return
    sid = df['station_id'].dropna().iloc[0].strip()
    df['timestamp_str'] = (df['year'].str.strip()+'-'+df['month'].str.strip()+'-'+
                           df['day'].str.strip()+' '+df['hour'].str.strip()+':'+df['minute'].str.strip())
    df['timestamp'] = pd.to_datetime(df['timestamp_str'], format='%Y-%m-%d %H:%M', errors='coerce')
    for col in _NUM_COLS:
        if col in df.columns: df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')
    df['data_completeness'] = (df[_FINAL_COLS].notna().all(axis=1) & df['timestamp'].notna()).astype(int)
    out = df[['timestamp']+_FINAL_COLS+['data_completeness']]
    out.to_csv(out_dir/f'{sid}.txt', index=False, na_rep='NaN')
    print(f'  {filepath.name} -> {sid}.txt ({len(out):,} rows, {out["data_completeness"].mean()*100:.1f}% complete)')

RAW_DATA_DIR = YSSY_CODE_DIR / 'RawDataC'
if RAW_DATA_DIR.exists():
    for rf in sorted(RAW_DATA_DIR.glob('*Data*.txt')):
        ingest_raw_bom_file(rf, RAW_TXT_DIR)
    print('Raw ingestion complete.')
else:
    print(f'RawDataC/ not found — skipping. Load from: {RAW_TXT_DIR}')

---
## Stage 1 — Station Merge
**Source:** `data-processing/merge_stations.py`

Outer merge on timestamp; newer file values take priority, gaps filled from older file.

> **Known issue in original:** hardcoded to YSRI. Generalised via `MERGE_PAIRS`.

In [ ]:
# SOURCE: data-processing/merge_stations.py

def load_station_txt(filepath):
    df = pd.read_csv(filepath, parse_dates=['timestamp'], na_values=['NaN'],
                     infer_datetime_format=True)
    for col in WEATHER_PARAMS:
        if col in df.columns and df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def merge_station_pair(f1, f2):
    df1, df2 = load_station_txt(f1), load_station_txt(f2)
    m = pd.merge(df1, df2, on='timestamp', how='outer', suffixes=('_f1','_f2'))
    for p in WEATHER_PARAMS:
        c1,c2 = f'{p}_f1',f'{p}_f2'
        if c2 in m.columns and c1 in m.columns: m[p] = m[c2].combine_first(m[c1])
        elif c2 in m.columns: m[p] = m[c2]
        elif c1 in m.columns: m[p] = m[c1]
        else: m[p] = np.nan
    m = m.drop(columns=[c for c in m.columns if c.endswith('_f1') or c.endswith('_f2')])
    meas = [p for p in WEATHER_PARAMS if p != 'aws_flag' and p in m.columns]
    m['data_completeness'] = m[meas].notna().all(axis=1).astype(int)
    return m.sort_values('timestamp').reset_index(drop=True)

older_files = {p[0] for p in MERGE_PAIRS}
station_data = {}
for f in sorted(RAW_TXT_DIR.glob('*.txt')):
    if f.name in older_files: continue
    pair = next((p for p in MERGE_PAIRS if p[1]==f.name), None)
    if pair:
        df = merge_station_pair(RAW_TXT_DIR/pair[0], f)
        sid, tag = pair[2], f'merged({pair[0]}+{f.name})'
    else:
        df, sid, tag = load_station_txt(f), f.stem, f.name
    df = df.set_index('timestamp').sort_index()
    station_data[sid] = df
    pct = df['data_completeness'].mean()*100 if 'data_completeness' in df.columns else float('nan')
    print(f'  {sid:8s}: {len(df):>7,} rows  {pct:5.1f}%  [{tag}]')
print(f'\n{len(station_data)} stations loaded.')

---
## Diagnostic A — Data Availability Plots (raw)
**Source:** `data-processing/plot_availability.py.py`

Monthly % availability per element, full-hour records only, before any interpolation.

In [ ]:
# SOURCE: data-processing/plot_availability.py.py

AVAIL_ELEMENTS = ['air_temp','dew_point','wind_speed','wind_dir','msl_pressure']
AVAIL_COLORS   = plt.cm.tab10(np.linspace(0, 0.5, len(AVAIL_ELEMENTS)))

for sid, df in station_data.items():
    df_fh = df[df.index.minute == 0].copy()
    if df_fh.empty: continue
    df_fh['ym'] = df_fh.index.to_period('M')
    fig, ax = plt.subplots(figsize=(14,4))
    for el, color in zip(AVAIL_ELEMENTS, AVAIL_COLORS):
        if el not in df_fh.columns: continue
        mon = df_fh.groupby('ym').agg(total=('air_temp','count'), avail=(el, lambda x: x.notna().sum()))
        mon['pct'] = mon['avail'] / mon['total'].replace(0, np.nan) * 100
        ax.plot(mon.index.to_timestamp(), mon['pct'], label=el.replace('_',' ').title(), color=color, marker='.', ms=3)
    ax.set_title(f'Station {sid} — Monthly Availability (full-hour, raw)')
    ax.set_ylabel('% available'); ax.set_ylim(0,105)
    ax.legend(loc='lower left', fontsize=8); ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/f'{sid}_availability_raw.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close()

---
## Stage 2 — Single-step Gap Fill (30 min)
**Source:** `data-processing/interpolateData0.5.py`

Fills isolated single NaN values with the average of two neighbours.

> **Known issue in original:** saves with `float_format='%.1f'`, losing precision. Kept in memory here.

In [ ]:
# SOURCE: data-processing/interpolateData0.5.py

def fill_single_step_gaps(series):
    s, na = series.copy(), series.isna()
    for i in range(1, len(series)-1):
        if not na.iloc[i-1] and na.iloc[i] and not na.iloc[i+1]:
            s.iloc[i] = (series.iloc[i-1]+series.iloc[i+1])/2.0
    return s

for sid, df in station_data.items():
    for el in INTERPOLATE_ELEMENTS:
        if el in df.columns: station_data[sid][el] = fill_single_step_gaps(df[el])
print('Single-step gap fill done.')

---
## Diagnostic B — Outage Distribution Plots
**Source:** `data-processing/outage_lengths.py`

Bar charts of consecutive-NaN run lengths for each element, after the 30-min fill.

> **Known issue in original:** uses deprecated `plt.cm.get_cmap()`. Fixed here.

In [ ]:
# SOURCE: data-processing/outage_lengths.py

OUTAGE_BINS = [
    (1,2,'30 min'),(2,3,'1 hr'),(3,5,'1.5-2 hr'),
    (5,13,'2-6 hr'),(13,49,'6-24 hr'),(49,337,'1-7 day'),
    (337,1441,'7d-1mo'),(1441,17521,'1mo-1yr'),(17521,float('inf'),'>1 yr'),
]
BIN_LABELS = [b[2] for b in OUTAGE_BINS]
OE = ['air_temp','dew_point','wind_speed','wind_dir','msl_pressure']

def outage_durations(series):
    out, run = [], 0
    for na in series.isna():
        if na: run+=1
        elif run: out.append(run); run=0
    if run: out.append(run)
    return out

def categorise(durations):
    counts = [0]*len(OUTAGE_BINS)
    for d in durations:
        for i,(lo,hi,_) in enumerate(OUTAGE_BINS):
            if lo<=d<hi: counts[i]+=1; break
    return counts

for sid, df in station_data.items():
    cols = [e for e in OE if e in df.columns]
    if not cols: continue
    x = np.arange(len(BIN_LABELS)); w = 0.8/len(cols)
    clrs = matplotlib.colormaps['viridis'](np.linspace(0,1,len(cols)))
    fig, ax = plt.subplots(figsize=(14,4))
    for j,(el,c) in enumerate(zip(cols,clrs)):
        ax.bar(x+j*w-0.4+w/2, categorise(outage_durations(df[el])), width=w, label=el, color=c, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(BIN_LABELS, fontsize=8)
    ax.set_title(f'{sid} — Outage Length Distribution (after 30-min fill)')
    ax.set_ylabel('Number of outages'); ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/f'{sid}_outage_dist.png', dpi=110, bbox_inches='tight')
    plt.show(); plt.close()

---
## Stage 3 — Wind Speed / Direction -> U / V Components
**Source:** `data-processing/convertWindComponents.py`

`u = -speed*sin(dir_rad)`,  `v = +speed*cos(dir_rad)` (meteorological convention)

In [ ]:
# SOURCE: data-processing/convertWindComponents.py

def wind_to_uv(df):
    if 'wind_speed' not in df.columns or 'wind_dir' not in df.columns: return df
    r = np.deg2rad(df['wind_dir'].astype(float))
    out = df.copy()
    out['u_component'] = -df['wind_speed']*np.sin(r)
    out['v_component'] =  df['wind_speed']*np.cos(r)
    mask = df['wind_speed'].isna() | df['wind_dir'].isna()
    out.loc[mask, ['u_component','v_component']] = np.nan
    return out.drop(columns=['wind_speed','wind_dir'])

def uv_to_wind(u, v):
    u,v = np.asarray(u,float), np.asarray(v,float)
    spd = np.sqrt(u**2+v**2)
    dirn = (np.rad2deg(np.arctan2(-u,-v))+360)%360
    return spd, np.where(spd>0.01, dirn, np.nan)

for sid in list(station_data): station_data[sid] = wind_to_uv(station_data[sid])
print('Wind -> U/V done.')
print(f'  {next(iter(station_data))} columns: {list(next(iter(station_data.values())).columns)}')

---
## Stage 4 — Spline Gap Fill (up to 2.5 hr)
**Source:** `data-processing/interpolate.py`

Fills NaN gaps of length <= `MAX_GAP_STEPS` with a local cubic spline (6 context points each side). After filling U/V, `wind_speed_recalc` and `wind_dir_recalc` are derived.

In [ ]:
# SOURCE: data-processing/interpolate.py

def _spline_fill_gap(series, gs, ge, ctx, k=3, s=1):
    n = len(series)
    before = series.iloc[max(0,gs-ctx):gs].dropna()
    after  = series.iloc[ge+1:min(n,ge+1+ctx)].dropna()
    x = list(range(max(0,gs-ctx),gs))[-len(before):] + list(range(ge+1,min(n,ge+1+ctx)))[:len(after)]
    y = list(before.values)+list(after.values)
    xu,idx = np.unique(x, return_index=True); yu = np.array(y)[idx]
    if len(xu)<k+1: return None
    try:
        sp=UnivariateSpline(xu,yu,k=k,s=s); xi=np.arange(gs,ge+1)
        return pd.Series(sp(xi), index=series.index[xi])
    except: return None

def spline_fill_series(series, max_gap=MAX_GAP_STEPS, ctx=6, k=SPLINE_ORDER, s=SPLINE_SMOOTH):
    out,na = series.copy(), series.isna().values
    i=0
    while i<len(na):
        if na[i]:
            j=i
            while j<len(na) and na[j]: j+=1
            if 0<(j-i)<=max_gap:
                filled=_spline_fill_gap(series,i,j-1,ctx,k,s)
                if filled is not None: out.loc[filled.index]=filled.values
            i=j
        else: i+=1
    return out

print('Applying spline interpolation (may take a few minutes)...')
for sid, df in station_data.items():
    for el in SPLINE_ELEMENTS:
        if el in df.columns: station_data[sid][el] = spline_fill_series(df[el])
    if 'u_component' in station_data[sid].columns:
        spd,dirn = uv_to_wind(station_data[sid]['u_component'].values,station_data[sid]['v_component'].values)
        station_data[sid]['wind_speed_recalc'] = spd
        station_data[sid]['wind_dir_recalc']   = dirn
    print(f'  {sid} done.')
print('Spline fill complete.')

### [Optional] Spline test on synthetic data
**Source:** `data-processing/interpolateTest.py`

In [ ]:
# SOURCE: data-processing/interpolateTest.py

np.random.seed(42); n=48; t=np.arange(n)
orig = np.sin(t/((n-1)/(2*np.pi)))+t/20+np.random.normal(0,0.3,n)
wg = orig.copy(); wg[20:30]=np.nan
filled = spline_fill_series(pd.Series(wg,index=t), max_gap=10, ctx=5, k=3, s=0.4)
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(t,orig,'k-',lw=1.5,label='Original')
ax.plot(t,wg,'go',ms=5,label='Known pts')
ax.plot(t,filled.values,'r--',lw=2,label='Spline fill')
ax.axvspan(20,29,color='yellow',alpha=0.3,label='Gap')
ax.set_title('Spline test (synthetic)'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.savefig(PLOTS_DIR/'spline_test.png',dpi=110,bbox_inches='tight')
plt.show(); plt.close()

---
## Diagnostic D — Concurrent Completeness (Post-Spline)
**Source:** `data-processing/plotTotalAvailability.py`

Per-element monthly availability + concurrent 24H rolling completeness, run after spline fill.
BELL and MTB excluded from pressure requirement.

In [ ]:
# SOURCE: data-processing/plotTotalAvailability.py — per-station plots

REQ_CONC  = ['air_temp','dew_point','u_component','msl_pressure']
ELEM_PLOT = ['air_temp','dew_point','u_component','wind_dir_recalc','msl_pressure']
DD_NO_P   = ['BELL','MTB']
_dc = matplotlib.colormaps['tab10'](np.linspace(0,0.9,len(ELEM_PLOT)))

def mon_pct(series):
    m = series.groupby(pd.Grouper(freq='ME')).agg(total='count',avail='sum')
    m['pct'] = np.where(m['total']>0, m['avail']/m['total']*100, 0.0)
    return m

for sid, df in station_data.items():
    no_p = sid in DD_NO_P
    req = [c for c in REQ_CONC if not(c=='msl_pressure' and no_p) and c in df.columns]
    dw = df.copy(); dw['_ok'] = dw[req].notna().all(axis=1).astype(int)
    mo = mon_pct(dw['_ok'])
    fig,ax = plt.subplots(figsize=(15,5))
    for i,el in enumerate(ELEM_PLOT):
        if el not in dw.columns: continue
        dw['_el']=dw[el].notna().astype(int)
        me=mon_pct(dw['_el'])
        if not me.empty: ax.plot(me.index,me['pct'],label=el.replace('_',' ').title(),color=_dc[i],lw=1.5)
    if not mo.empty: ax.scatter(mo.index,mo['pct'],label='Overall Complete',color='black',marker='o',s=30,zorder=5)
    ax.set_title(f'Monthly Availability (post-spline) — {sid} (Since {START_YEAR})',fontsize=13)
    ax.set_xlabel('Month'); ax.set_ylabel('% Available'); ax.set_ylim(0,105)
    ax.legend(loc='best',fontsize=8); ax.grid(True,linestyle='--',linewidth=0.5,alpha=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/f'{sid}_availability_post_spline.png',dpi=110,bbox_inches='tight')
    plt.show(); plt.close(fig)
print('Per-station post-spline plots done.')

In [ ]:
# Concurrent 24H rolling window — SOURCE: plotTotalAvailability.py

CON_WIN = 48
con_s = {}
for sid,df in station_data.items():
    no_p = sid in DD_NO_P
    req = [c for c in REQ_CONC if not(c=='msl_pressure' and no_p) and c in df.columns]
    if req: con_s[sid] = df[req].notna().all(axis=1).astype(int)

if con_s:
    cdf = pd.DataFrame(con_s).fillna(0).astype(int)
    aok = cdf.all(axis=1).astype(int)
    fi = pd.date_range(start=aok.index.min(),end=aok.index.max(),freq='30min')
    aok = aok.reindex(fi,fill_value=0)
    wok = (aok.rolling(window=CON_WIN,min_periods=CON_WIN).sum()==CON_WIN).astype(int)
    mc = wok.groupby(pd.Grouper(freq='ME')).agg(total='count',good='sum')
    mc['pct'] = np.where(mc['total']>0, mc['good']/mc['total']*100, 0.0)
    fig,ax=plt.subplots(figsize=(15,5))
    ax.plot(mc.index,mc['pct'],marker='o',lw=1.5)
    ax.set_title(f'Monthly % Concurrent 24H Completeness\n({len(con_s)} Stations, Since {START_YEAR})',fontsize=13)
    ax.set_xlabel('Month'); ax.set_ylabel('% Timesteps with 24H Concurrent Data'); ax.set_ylim(0,105)
    ax.grid(True,linestyle='--',linewidth=0.5,alpha=0.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR/'concurrent_completeness_24H.png',dpi=110,bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Concurrent 24H: {int(wok.sum()):,}/{len(wok):,} timesteps')
else: print('No concurrent data.')

---
## Stage 5 — Drop Columns + Crop to Year Range
**Sources:** `data-processing/drop_columns.py` + `data-processing/crop_to_years.py`

> **Bug fixed:** `drop_columns.py` second config block overwrites first — fixed inline here.

> **Bug fixed:** `crop_to_years.py` had `END_YEAR = START_YEAR = 2000` — `END_YEAR` now 2024.

In [ ]:
# SOURCE: data-processing/drop_columns.py  +  data-processing/crop_to_years.py

for sid, df in station_data.items():
    df = df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])
    df = df[(df.index.year>=START_YEAR)&(df.index.year<=END_YEAR)]
    chk = [c for c in ['air_temp','dew_point','msl_pressure','wind_speed_recalc'] if c in df.columns]
    df['data_completeness'] = df[chk].notna().all(axis=1).astype(int)
    station_data[sid] = df
    print(f'  {sid}: {len(df):>7,} rows  {START_YEAR}-{END_YEAR}  {df["data_completeness"].mean()*100:.1f}%')
print('Column drop + year crop done.')

---
## Diagnostic C — Missing Timestamp Check
**Source:** `data-processing/find_missing_timestamp.py`

> **Known issue in original:** hardcoded stations, no `__main__` guard. Wrapped in a function here.

In [ ]:
# SOURCE: data-processing/find_missing_timestamp.py

def check_missing_timestamps(station_id, ref_id=None, freq='30min'):
    if station_id not in station_data: print(f'{station_id} not found'); return
    expected = pd.date_range(start=f'{START_YEAR}-01-01',end=f'{END_YEAR}-12-31 23:30',freq=freq)
    missing = sorted(set(expected)-set(station_data[station_id].index))
    print(f'{station_id} vs grid: {len(missing)} missing timestamps')
    for ts in missing[:10]: print(f'  {ts}')
    if len(missing)>10: print(f'  ... and {len(missing)-10} more')
    if ref_id and ref_id in station_data:
        mr = sorted(set(station_data[ref_id].index)-set(station_data[station_id].index))
        print(f'{station_id} vs {ref_id}: {len(mr)} in ref but not target')

sids=list(station_data.keys())
check_missing_timestamps(TARGET_STATION, ref_id=sids[1] if len(sids)>1 else None)

---
## Parquet Checkpoint — Save All Processed Stations

Save every station to compressed Parquet. Reload block below for fresh kernels.

In [ ]:
for sid, df in station_data.items():
    out = PARQUET_DIR/f'{sid}.parquet'
    df.reset_index().to_parquet(out, index=False)
    print(f'  Saved: {out.name}  ({len(df):,} rows)')
print(f'\nParquet files in: {PARQUET_DIR.resolve()}')

In [ ]:
# Reload from Parquet (uncomment for fresh kernel)
# station_data = {}
# for pq in sorted(PARQUET_DIR.glob('*.parquet')):
#     df = pd.read_parquet(pq); df['timestamp'] = pd.to_datetime(df['timestamp'])
#     station_data[pq.stem] = df.set_index('timestamp').sort_index()
# print(f'Reloaded {len(station_data)} stations.')
print('station_data in memory — skipping reload.')

---
## Stage 6 — Feature Engineering & ML Dataset
**Source:** `data-processing/preprocess_data.py`

Each row = one anchor time *t*. Targets = YSSY U/V at *t*+1 ... *t*+48.

| Feature group | Description |
|---|---|
| `YSSY_{feat}_lag0-47` | YSSY all 48 lags (24 hr history) |
| `{SID}_{feat}_lag0-11` | Other stations dense lags (6 hr) |
| `{SID}_{feat}_lag12,14,...,46` | Other stations sparse lags (6-23 hr) |
| `PG_{S1}_{S2}_{t,t-6hr,t-12hr,t-24hr}` | Pressure gradients at 4 time points (3 pairs) |
| `PG_..._trend_t_vs_t_minus_{6,12}hr` | Gradient trend over 6/12 hr |
| `{stn}_{param}_deriv_*` | 3-hr time derivatives (8 periods) + t vs t-6hr/t-12hr |
| `BELL_v_deriv_*` | BELL v-component rate-of-change (12 pts x 2 lags) |
| `sin/cos_time_of_day`, `sin/cos_day_of_year` | Cyclical time (48 half-hours/day) |

Stations: `BELL, MTB, YBTH, YCNK, YSBK, YSCN, YSNW, YSRI, YSSY, YSWG`  
BELL and MTB have `msl_pressure` excluded.

> **Bug fixed:** `preprocess_data.py` lines 21-23: `TIMESTEP_MIN` + `UTES = 30` instead of
> `TIMESTEP_MINUTES = 30` (NameError). Fixed here.
>
> **Proportions fixed:** original placeholder 8%/1%/1% -> 80%/10%/10%.
>
> **Implementation:** vectorised `pandas.shift()` instead of original slow row-by-row loop.

In [ ]:
# SOURCE: data-processing/preprocess_data.py

_PG = [('YBTH','YSSY','PG_YBTH_YSSY'),('YSSY','YCNK','PG_YSSY_YCNK'),('YSSY','YSNW','PG_YSSY_YSNW')]
_S3H, _N3H = 6, 8

def build_ml_dataset(stn_data, tgt, lb, fc):
    idx = stn_data[tgt].index
    for sid in STATIONS:
        if sid in stn_data: idx = idx.intersection(stn_data[sid].index)
    idx = idx.sort_values()
    print(f'Common index: {idx[0]} -> {idx[-1]}  ({len(idx):,} steps)')
    aln = {sid: stn_data[sid].reindex(idx) for sid in STATIONS if sid in stn_data}
    fd = {}

    # A. Lag features
    for sid, df in aln.items():
        feats = [f for f in ALL_FILE_FEATURES if not(f=='msl_pressure' and sid in STATIONS_NO_PRESSURE) and f in df.columns]
        lags = range(lb) if sid==tgt else list(range(12))+list(range(12,lb,2))
        for lag in lags:
            for col in feats: fd[f'{sid}_{col}_lag{lag}'] = df[col].shift(lag)

    # B. Cyclical time
    hh = idx.hour*2+idx.minute//30
    fd['sin_time_of_day'] = np.sin(2*np.pi*hh/48.0)
    fd['cos_time_of_day'] = np.cos(2*np.pi*hh/48.0)
    diy = np.where(idx.is_leap_year,366,365)
    fd['sin_day_of_year'] = np.sin(2*np.pi*idx.dayofyear/diy)
    fd['cos_day_of_year'] = np.cos(2*np.pi*idx.dayofyear/diy)

    # C. Pressure gradients
    pgs = {'t':0,'t_minus_6hr':12,'t_minus_12hr':24,'t_minus_24hr':48}
    for s1,s2,lbl in _PG:
        g={}
        for tl,sh in pgs.items():
            gr=aln[s1]['msl_pressure'].shift(sh)-aln[s2]['msl_pressure'].shift(sh)
            fd[f'{lbl}_{tl}']=gr; g[tl]=gr
        fd[f'{lbl}_trend_t_vs_t_minus_6hr'] =g['t']-g['t_minus_6hr']
        fd[f'{lbl}_trend_t_vs_t_minus_12hr']=g['t']-g['t_minus_12hr']

    # D. Time derivatives
    for sid, df in aln.items():
        for p in ['air_temp','dew_point','msl_pressure']:
            if p not in df.columns or (p=='msl_pressure' and sid in STATIONS_NO_PRESSURE): continue
            s=df[p]
            for per in range(_N3H):
                es,ss=per*_S3H,(per+1)*_S3H; eh,sh=per*3,(per+1)*3
                fd[f'{sid}_{p}_deriv_t_minus_{eh}hr_vs_t_minus_{sh}hr']=s.shift(es)-s.shift(ss)
            fd[f'{sid}_{p}_deriv_t_vs_t_minus_6hr'] =s-s.shift(12)
            fd[f'{sid}_{p}_deriv_t_vs_t_minus_12hr']=s-s.shift(24)

    # E. BELL v derivatives
    if 'BELL' in aln and 'v_component' in aln['BELL'].columns:
        bv=aln['BELL']['v_component']
        for so in range(12):
            lb2=f'{so*0.5:.1f}'
            fd[f'BELL_v_deriv_eval_at_t_minus_{lb2}hr_lag_0.5hr']=bv.shift(so)-bv.shift(so+1)
            fd[f'BELL_v_deriv_eval_at_t_minus_{lb2}hr_lag_1.0hr']=bv.shift(so)-bv.shift(so+2)

    feat_df=pd.DataFrame(fd,index=idx)
    tgt_df=aln[tgt]
    td={}
    for h in range(1,fc+1):
        td[f'{tgt}_u_forecast_t_plus_{h}']=tgt_df['u_component'].shift(-h)
        td[f'{tgt}_v_forecast_t_plus_{h}']=tgt_df['v_component'].shift(-h)
    ts=pd.Series(idx,index=idx,name=TIMESTAMP_COL)
    ml=pd.concat([ts,feat_df,pd.DataFrame(td,index=idx)],axis=1).dropna()
    print(f'ML samples: {len(ml):,}  |  columns: {ml.shape[1]}')
    return ml

print('Building ML dataset...')
ml_df = build_ml_dataset(station_data, TARGET_STATION, LOOKBACK_STEPS, FORECAST_STEPS)

In [ ]:
# Chronological 80/10/10 split — original had placeholder 8%/1%/1%

ml_df = ml_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
n = len(ml_df); n_tr=int(n*TRAIN_PROP); n_va=int(n*VAL_PROP)
splits={'training':ml_df.iloc[:n_tr],'validation':ml_df.iloc[n_tr:n_tr+n_va],'test':ml_df.iloc[n_tr+n_va:]}
ML_PARQUET_DIR = PARQUET_DIR/'ml'; ML_PARQUET_DIR.mkdir(exist_ok=True)
for name,df in splits.items():
    out=ML_PARQUET_DIR/f'{name}_dataset.parquet'; df.to_parquet(out,index=False)
    t0=pd.to_datetime(df[TIMESTAMP_COL].iloc[0]).strftime('%Y-%m-%d')
    t1=pd.to_datetime(df[TIMESTAMP_COL].iloc[-1]).strftime('%Y-%m-%d')
    print(f'  {name:10s}: {len(df):>6,} samples  [{t0} -> {t1}]')

---
## Stage 7A — Quick Model (fixed hyperparameters)
**Source:** `model/YSSY_winds_24hr.py`

Sanity-check model: 10 estimators, 10% data subsample. Verify the pipeline works before committing to a long training run.

In [ ]:
# SOURCE: model/YSSY_winds_24hr.py

def load_ml_parquet(path):
    df=pd.read_parquet(path)
    tc=sorted([c for c in df.columns if c.startswith(f'{TARGET_STATION}_u_forecast_t_plus_') or
               c.startswith(f'{TARGET_STATION}_v_forecast_t_plus_')],
              key=lambda x:(int(x.rsplit('_',1)[-1]),x.split('_')[1]))
    fc=[c for c in df.columns if c!=TIMESTAMP_COL and c not in tc]
    return df,df[fc],df[tc],pd.to_datetime(df[TIMESTAMP_COL]),tc,fc

train_full,X_train,y_train,ts_train,TARGET_COLS,FEATURE_COLS = load_ml_parquet(ML_PARQUET_DIR/'training_dataset.parquet')
val_full,  X_val,  y_val,  ts_val,  _,_             = load_ml_parquet(ML_PARQUET_DIR/'validation_dataset.parquet')
test_full, X_test, y_test, ts_test, _,_             = load_ml_parquet(ML_PARQUET_DIR/'test_dataset.parquet')

QP={'objective':'regression_l2','metric':'l2','n_estimators':10,'learning_rate':0.05,
    'num_leaves':10,'max_depth':10,'min_child_samples':10,'subsample':0.8,
    'colsample_bytree':0.7,'reg_alpha':0.5,'reg_lambda':0.1,'random_state':42,'n_jobs':-1,'verbose':-1}
Xq=X_train.sample(frac=0.1,random_state=42); yq=y_train.loc[Xq.index]
mq=MultiOutputRegressor(lgb.LGBMRegressor(**QP))
t0=time.time(); mq.fit(Xq,yq)
print(f'Quick model: {time.time()-t0:.1f}s  ({Xq.shape[0]:,} samples, {yq.shape[1]} targets)')
joblib.dump(mq, MODEL_DIR/'yssy_quick_model.joblib')
print('Saved: models/yssy_quick_model.joblib')

---
## Stage 7B — Full Tuned Model
**Source:** `model/YSSY_LightGMB.py`

`MultiOutputRegressor` with Optuna-tuned hyperparameters on full training data. This is the main model.

In [ ]:
# SOURCE: model/YSSY_LightGMB.py -> train_final_lgbm_model()

model=MultiOutputRegressor(lgb.LGBMRegressor(**LGBM_PARAMS))
print(f'Training: {X_train.shape[0]:,} samples x {X_train.shape[1]} features -> {y_train.shape[1]} targets')
print('(This may take several minutes.)')
t0=time.time(); model.fit(X_train,y_train)
print(f'Done in {time.time()-t0:.0f}s.')
joblib.dump(model, MODEL_DIR/'yssy_wind_model.joblib')
print('Saved: models/yssy_wind_model.joblib')

---
## Stage 7C — [Optional] Per-target Individual Models
**Sources:** `model/tune_single_model.py` + `model/train_final_individual.py`

One LightGBM per forecast target (96 models). Commented out by default — takes 30-60 min.

In [ ]:
# SOURCE: model/train_final_individual.py  (uncomment to run)

# INDIV_PARAMS={'learning_rate':0.01,'num_leaves':300,'max_depth':40,
#     'min_child_samples':100,'subsample':0.7,'colsample_bytree':0.7,
#     'reg_alpha':0.5,'reg_lambda':0.1,'n_estimators':4000,
#     'objective':'regression_l2','metric':'l2','random_state':42,'n_jobs':-1,'verbose':-1}
# INDIV_DIR=MODEL_DIR/'individual_models'; INDIV_DIR.mkdir(exist_ok=True)
# for tc in TARGET_COLS:
#     m=lgb.LGBMRegressor(**INDIV_PARAMS)
#     m.fit(X_train,y_train[tc],eval_set=[(X_val,y_val[tc])],
#           callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(period=-1)])
#     joblib.dump(m, INDIV_DIR/f'{tc}.joblib')
#     print(f'  {tc}  best_iter={m.best_iteration_}')
print('Per-target block commented out.')

---
## Stage 8 — Evaluation
**Source:** `model/evaluate_all_24hr_models.py`

In [ ]:
# SOURCE: model/evaluate_all_24hr_models.py

preds=model.predict(X_test)
print(f'Overall Test MAE: {mean_absolute_error(y_test,preds):.4f} kt')
print(f'Overall Test MSE: {mean_squared_error(y_test,preds):.4f} kt2')

mae_u,mae_v=[],[]
for step in range(1,FORECAST_STEPS+1):
    iu=TARGET_COLS.index(f'{TARGET_STATION}_u_forecast_t_plus_{step}')
    iv=TARGET_COLS.index(f'{TARGET_STATION}_v_forecast_t_plus_{step}')
    mae_u.append(mean_absolute_error(y_test.iloc[:,iu],preds[:,iu]))
    mae_v.append(mean_absolute_error(y_test.iloc[:,iv],preds[:,iv]))

hours=[h*0.5 for h in range(1,FORECAST_STEPS+1)]
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(hours,mae_u,'b-o',ms=3,lw=1.5,label='U MAE')
ax.plot(hours,mae_v,'r-o',ms=3,lw=1.5,label='V MAE')
ax.set_xlabel('Forecast horizon (hours)'); ax.set_ylabel('MAE (knots)')
ax.set_title('Test Set — MAE vs Horizon'); ax.legend(); ax.grid(True,alpha=0.4)
ax.set_xlim(0,24); ax.xaxis.set_major_locator(mticker.MultipleLocator(3))
plt.tight_layout(); plt.savefig(PLOTS_DIR/'mae_vs_horizon.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
rows=[(f'{h} hr',mae_u[h*2-1],mae_v[h*2-1]) for h in [1,3,6,12,24]]
s=pd.DataFrame(rows,columns=['Horizon','U MAE (kt)','V MAE (kt)'])
s['Mean MAE (kt)']=(s['U MAE (kt)']+s['V MAE (kt)'])/2
display(s.set_index('Horizon').round(4))

---
## Stage 9 — Visual Display
**Sources:** `model/plot_samples_24hr.py` + `model/plot_test_samples.py`

Gray = past observed, Blue = actual future, Red dashed = forecast.

In [ ]:
# SOURCE: model/plot_samples_24hr.py / model/plot_test_samples.py

def plot_forecast_sample(anchor_ts, mdl, row, tgt_cols, feat_cols, idx=''):
    ul=sorted([c for c in feat_cols if c.startswith(f'{TARGET_STATION}_u_component_lag')],key=lambda x:int(x.split('lag')[-1]))
    vl=sorted([c for c in feat_cols if c.startswith(f'{TARGET_STATION}_v_component_lag')],key=lambda x:int(x.split('lag')[-1]))
    pu,pv=row[ul].values[::-1],row[vl].values[::-1]
    np_=len(ul)
    pt=[anchor_ts-timedelta(minutes=30*(np_-1-i)) for i in range(np_)]
    ft=[anchor_ts+timedelta(minutes=30*h) for h in range(1,FORECAST_STEPS+1)]
    au=np.array([row.get(f'{TARGET_STATION}_u_forecast_t_plus_{h}',np.nan) for h in range(1,FORECAST_STEPS+1)])
    av=np.array([row.get(f'{TARGET_STATION}_v_forecast_t_plus_{h}',np.nan) for h in range(1,FORECAST_STEPS+1)])
    Xs=pd.DataFrame([row[feat_cols].values],columns=feat_cols)
    pred=mdl.predict(Xs)[0]
    pu2=np.array([pred[tgt_cols.index(f'{TARGET_STATION}_u_forecast_t_plus_{h}')] for h in range(1,FORECAST_STEPS+1)])
    pv2=np.array([pred[tgt_cols.index(f'{TARGET_STATION}_v_forecast_t_plus_{h}')] for h in range(1,FORECAST_STEPS+1)])
    ps,pd_=uv_to_wind(pu,pv); as_,ad=uv_to_wind(au,av); fs,fd=uv_to_wind(pu2,pv2)
    fig,(a1,a2)=plt.subplots(2,1,figsize=(16,8),sharex=True)
    fig.suptitle(f'YSSY Forecast — {anchor_ts.strftime("%Y-%m-%d %H:%M UTC")}',fontsize=13)
    a1.plot(pt,ps,'o-',color='dimgray',lw=1.5,ms=4,label='Observed (input)')
    a1.plot(ft,as_,'s-',color='steelblue',lw=1.5,ms=4,label='Observed (future)')
    a1.plot(ft,fs,'x--',color='crimson',lw=2,ms=5,label='Forecast')
    a1.axvline(anchor_ts,color='k',ls=':',lw=1,alpha=0.5)
    a1.set_ylabel('Wind speed (kt)'); a1.legend(loc='upper left',fontsize=8)
    a1.set_ylim(bottom=0); a1.grid(True,alpha=0.3)
    a2.scatter(pt,pd_,color='dimgray',s=18,label='Observed (input)')
    a2.scatter(ft,ad,color='steelblue',s=18,label='Observed (future)')
    a2.scatter(ft,fd,color='crimson',s=30,marker='x',label='Forecast')
    a2.axvline(anchor_ts,color='k',ls=':',lw=1,alpha=0.5)
    a2.set_ylabel('Wind dir (deg)'); a2.set_ylim(0,360); a2.set_yticks([0,90,180,270,360])
    a2.legend(loc='upper left',fontsize=8); a2.grid(True,alpha=0.3); a2.set_xlabel('Time')
    a2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d-%b'))
    a2.xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.tight_layout()
    fname=PLOTS_DIR/f'forecast_{anchor_ts.strftime("%Y%m%d_%H%M")}{idx}.png'
    plt.savefig(fname,dpi=120,bbox_inches='tight'); plt.show(); plt.close(fig)
    print(f'Saved: {fname.name}')

print('Plot function ready.')

In [ ]:
# Random sample loop — SOURCE: model/plot_samples_24hr.py

random.seed(42)
for i,ix in enumerate(random.sample(range(len(test_full)), min(5,len(test_full)))):
    row=test_full.iloc[ix]; ts=pd.to_datetime(row[TIMESTAMP_COL])
    print(f'\n--- Sample {i+1}: {ts} ---')
    plot_forecast_sample(ts,model,row,TARGET_COLS,FEATURE_COLS,idx=f'_s{i+1}')
print(f'\nPlots in: {PLOTS_DIR.resolve()}')

In [ ]:
# Date-range loop — SOURCE: model/plot_samples_24hr.py / model/plot_test_samples.py

LOOP_START,LOOP_END='2024-01-01 00:00','2024-01-02 00:00'
tsi=pd.to_datetime(test_full[TIMESTAMP_COL])
for ts in pd.date_range(start=LOOP_START,end=LOOP_END,freq='6h'):
    match=test_full[tsi==ts]
    if match.empty: continue
    plot_forecast_sample(ts,model,match.iloc[0],TARGET_COLS,FEATURE_COLS,idx=f'_{ts.strftime("%Y%m%d_%H%M")}')
print('Date-range loop done.')